In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("C:\\Users\\karth\\Downloads\\sample_Zends_synthetic_dataset.csv")

In [3]:
df.head()

,text,service_type,sentiment
0,The Billing speed is amazing.,Billing & Payments,Positive
1,Very satisfied with the Customer Support perfo...,Customer Support,Positive
2,Customer experience with 5G Service is great.,Service Activation,Positive
3,Great experience with the Fiber Broadband.,Broadband Service,Positive
4,Really impressed with the Customer Support.,Customer Support,Positive


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   text          10000 non-null  object
 1   service_type  10000 non-null  object
 2   sentiment     10000 non-null  object
dtypes: object(3)
memory usage: 234.5+ KB


In [5]:
df.describe()

,text,service_type,sentiment
count,10000,10000,10000
unique,150,5,3
top,Customer service regarding Fiber Broadband is ...,Broadband Service,Negative
freq,93,2032,3341


In [6]:
print(df.shape)

(10000, 3)


In [7]:
print(df.columns)

Index(['text', 'service_type', 'sentiment'], dtype='object')


In [8]:
print(df.isnull().sum())

text            0
service_type    0
sentiment       0
dtype: int64


In [9]:
print(df['sentiment'].value_counts())

sentiment
Negative    3341
Neutral     3340
Positive    3319
Name: count, dtype: int64


In [10]:
from sklearn.preprocessing import LabelEncoder

# Label Encoding
encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['sentiment'])

# Check encoded labels
print(df[['sentiment', 'label']].head())

# Label mapping
print(dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

  sentiment  label
0  Positive      2
1  Positive      2
2  Positive      2
3  Positive      2
4  Positive      2
{'Negative': 0, 'Neutral': 1, 'Positive': 2}


In [11]:
from sklearn.model_selection import train_test_split

X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 8000
Testing samples: 2000


In [25]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [13]:
train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=128
)

In [14]:
import numpy as np

train_labels = np.array(y_train)
test_labels = np.array(y_test)

In [26]:
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

print("Model loaded successfully")

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully


In [16]:
import tensorflow as tf

print(tf.__version__)

2.15.0


In [27]:
from transformers import BertTokenizer, BertForSequenceClassification
print("Transformers library imported successfully")

Transformers library imported successfully


In [28]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

print("Model loaded")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded


In [29]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization complete")

Tokenization complete


In [20]:
import torch

class TelecomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item['labels'] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [21]:
train_dataset = TelecomDataset(
    train_encodings,
    list(y_train)
)

test_dataset = TelecomDataset(
    test_encodings,
    list(y_test)
)

print("Datasets ready")

Datasets ready


In [30]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

print("Model loaded successfully")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully


In [31]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=10
)

trainer doesn't work so mannual training the BERT

In [34]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8
)

print("DataLoader ready")

DataLoader ready


In [35]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [36]:
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [37]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [38]:
from tqdm import tqdm

model.train()

for epoch in range(2):

    total_loss = 0

    progress_bar = tqdm(train_loader)

    for batch in progress_bar:

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        total_loss += loss.item()

        loss.backward()

        optimizer.step()

        progress_bar.set_description(
            f"Epoch {epoch+1}"
        )

        progress_bar.set_postfix(
            loss=loss.item()
        )

    avg_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} Loss: {avg_loss}")

Epoch 1: 100%|██████████| 1000/1000 [09:10<00:00,  1.82it/s, loss=0.000488]



Epoch 1 Loss: 0.04230909811335732


Epoch 2: 100%|██████████| 1000/1000 [09:19<00:00,  1.79it/s, loss=0.000193]


Epoch 2 Loss: 0.00031756090330600274


In [39]:
from sklearn.metrics import accuracy_score
import numpy as np

model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(true_labels, predictions)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [40]:
from sklearn.metrics import classification_report

print(classification_report(true_labels, predictions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       668
           1       1.00      1.00      1.00       668
           2       1.00      1.00      1.00       664

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [41]:
model.save_pretrained("./sentiment_model")

tokenizer.save_pretrained("./sentiment_model")

print("Model saved successfully")

Model saved successfully


In [42]:
label_map = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

def predict_sentiment(text):

    model.eval()

    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        prediction = torch.argmax(logits, dim=1).item()

    return label_map[prediction]

In [43]:
text1 = "The network issue has not been resolved for 3 days"

text2 = "Customer support was helpful and solved my issue quickly"

text3 = "Internet speed is average but manageable"

print(predict_sentiment(text1))
print(predict_sentiment(text2))
print(predict_sentiment(text3))

Negative
Positive
Neutral
